# 실습 8. 신뢰구간, 효과크기, 교란

사회약학 연구방법 노트의 '실습 8. 신뢰구간, 효과크기, 교란'에 딸린 노트북입니다. MASS 패키지의 birthwt 자료(출생아와 산모 189명)로 '임신 중 흡연은 저체중아 출산과 관련이 있는가, 인종을 고려하면 어떻게 달라지는가'를 신뢰구간과 효과 지표, 일반화 선형모형과 우도비 검정, 층화 분석, 가변수와 다변수 모형으로 차례로 분석합니다. 셀을 위에서부터 차례로 실행하세요. 설명과 출력 읽는 법은 사이트의 실습 8에 있습니다.


## a. 신뢰구간과 효과크기

이 실습이 쓰는 패키지를 불러와 버전을 확인한 뒤 birthwt 자료를 읽습니다. 자료는 인터넷에서 읽어 오므로 인터넷 연결이 필요합니다.

### 셀 1. 패키지 불러오기와 버전 확인

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import scipy, statsmodels, matplotlib

print("pandas     ", pd.__version__)
print("numpy      ", np.__version__)
print("scipy      ", scipy.__version__)
print("statsmodels", statsmodels.__version__)
print("matplotlib ", matplotlib.__version__)

### 셀 2. 데이터 불러오기와 변수 만들기

In [ ]:
url = ("https://vincentarelbundock.github.io/Rdatasets/"
       "csv/MASS/birthwt.csv")
df = pd.read_csv(url)
df["ptl_any"] = (df["ptl"] > 0).astype(int)  # 이전 조산 1회 이상
df["lwt_kg"] = df["lwt"] * 0.4536            # 파운드 → kg
df["race_f"] = pd.Categorical(
    df["race"].map({1: "White", 2: "Black", 3: "Other"}),
    categories=["White", "Black", "Other"])
print(df.shape)
df[["low", "smoke", "race", "age", "lwt_kg", "ptl_any", "ht"]].head(3)

### 셀 3. 비율의 95% 신뢰구간

In [ ]:
from statsmodels.stats.proportion import proportion_confint

rows = []
for count, n in [(59, 189), (7, 12)]:
    for method in ["normal", "wilson", "beta"]:
        lo, hi = proportion_confint(count, n, method=method)
        rows.append([f"{count}/{n}", method, count / n, lo, hi])
cols = ["data", "method", "p", "lower", "upper"]
pd.DataFrame(rows, columns=cols).round(3)

### 셀 4. 평균의 95% 신뢰구간

In [ ]:
bwt = df["bwt"]
n = len(bwt)
mean, sd = bwt.mean(), bwt.std()   # std()는 n - 1로 나눈 표준편차
se = sd / np.sqrt(n)
t_crit = stats.t.ppf(0.975, n - 1)  # 자유도 188인 t 분포의 97.5% 점
print(f"mean = {mean:.1f}, SD = {sd:.1f}, SE = {se:.2f}")
print(f"t = {t_crit:.3f}, half-width = {t_crit * se:.1f}")
print(f"95% CI = {mean - t_crit * se:.1f} to {mean + t_crit * se:.1f}")

### 셀 5. 흡연과 저체중아의 2×2 표

In [ ]:
tab = pd.crosstab(df["smoke"], df["low"]).loc[[1, 0], [1, 0]]
print(tab)
(a, b), (c, d) = tab.to_numpy()
p1, p0 = a / (a + b), c / (c + d)
print(f"risk: smoker = {p1:.3f}, nonsmoker = {p0:.3f}")

### 셀 6. 위험차, 상대위험도, 오즈비, NNH

In [ ]:
from statsmodels.stats.contingency_tables import Table2x2
from statsmodels.stats.proportion import confint_proportions_2indep

def show(name, est, ci):
    print(f"{name:3s} = {est:6.3f} (95% CI {ci[0]:.3f} to {ci[1]:.3f})")

t22 = Table2x2(tab.to_numpy())
rd_ci = confint_proportions_2indep(a, a + b, c, c + d,
                                   compare="diff", method="wald")
show("RD", p1 - p0, rd_ci)
show("RR", t22.riskratio, t22.riskratio_confint())
show("OR", t22.oddsratio, t22.oddsratio_confint())
show("NNH", 1 / (p1 - p0), (1 / rd_ci[1], 1 / rd_ci[0]))

### 셀 7. 부트스트랩 신뢰구간

In [ ]:
rng = np.random.default_rng(2026)   # 시드를 고정해 같은 결과 재현
y = df["low"].to_numpy()
x = df["smoke"].to_numpy()
boot_rr = []
for _ in range(2000):
    i = rng.integers(0, len(y), size=len(y))   # 189명 복원추출
    yb, xb = y[i], x[i]
    boot_rr.append(yb[xb == 1].mean() / yb[xb == 0].mean())
boot_rr = np.array(boot_rr)
lo, hi = np.percentile(boot_rr, [2.5, 97.5])
print(f"bootstrap RR: median {np.median(boot_rr):.3f}, "
      f"95% CI {lo:.3f} to {hi:.3f}")

### 셀 8. 부트스트랩 분포 그래프

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 3))
plt.hist(boot_rr, bins=40, color="lightsteelblue", edgecolor="white")
for v in (lo, hi):
    plt.axvline(v, ls="--", color="k")          # 백분위수 구간
plt.axvline(t22.riskratio, color="tab:orange")  # 표본의 RR
plt.xlabel("Risk ratio in bootstrap samples")
plt.ylabel("Count")
plt.show()

## b. 일반화 선형모형과 우도비 검정

### 셀 9. 로지스틱 회귀를 GLM으로 적합하기

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

fam = sm.families
m_logit = smf.glm("low ~ smoke", data=df,
                  family=fam.Binomial()).fit()
print(m_logit.summary())

### 셀 10. 계수를 오즈비로 바꾸기

In [ ]:
or_tab = np.exp(m_logit.conf_int())
or_tab.columns = ["2.5%", "97.5%"]
or_tab.insert(0, "OR", np.exp(m_logit.params))
or_tab.round(3)

### 셀 11. 연결함수와 분포가 다른 네 모형

In [ ]:
f = "low ~ smoke"
fits = {
    "logit": smf.glm(f, df, family=fam.Binomial()).fit(),
    "log-binomial": smf.glm(
        f, df, family=fam.Binomial(fam.links.Log())).fit(),
    "Poisson": smf.glm(f, df, family=fam.Poisson()).fit(),
    "Poisson robust": smf.glm(
        f, df, family=fam.Poisson()).fit(cov_type="HC0"),
}

### 셀 12. 흡연의 효과를 네 모형으로 비교

In [ ]:
rows = []
for name, r in fits.items():
    lo, hi = np.exp(r.conf_int().loc["smoke"])
    rows.append([name, np.exp(r.params["smoke"]), r.bse["smoke"],
                 lo, hi])
cols = ["model", "exp(b)", "SE(b)", "2.5%", "97.5%"]
pd.DataFrame(rows, columns=cols).round(3)

### 셀 13. 로그우도, 이탈도, AIC

In [ ]:
m0 = smf.glm("low ~ 1", df, family=fam.Binomial()).fit()  # 절편만
for name, r in [("null", m0), ("smoke", m_logit)]:
    print(f"{name:5s}: llf = {r.llf:.3f}, deviance = {r.deviance:.3f},"
          f" df_model = {r.df_model:.0f}, AIC = {r.aic:.3f}")

### 셀 14. 우도비 검정을 직접 계산하기

In [ ]:
lr = 2 * (m_logit.llf - m0.llf)
print(f"LR    chi2 = {lr:.3f}, P = {stats.chi2.sf(lr, 1):.4f}")
z = m_logit.tvalues["smoke"]
print(f"Wald  chi2 = {z**2:.3f}, P = {m_logit.pvalues['smoke']:.4f}")
sc = stats.chi2_contingency(tab, correction=False)
print(f"Score chi2 = {sc.statistic:.3f}, P = {sc.pvalue:.4f}")

### 셀 15. 보정 모형에서 로그-이항 모형이 실패할 때

In [ ]:
f_adj = "low ~ smoke + C(race) + age + lwt_kg + ptl_any + ht"
lb = smf.glm(f_adj, df, family=fam.Binomial(fam.links.Log())).fit()
print("converged     :", lb.converged)
print("log-likelihood:", lb.llf)
print("max fitted p  :", lb.fittedvalues.max())

### 셀 16. 수정 포아송 회귀로 보정 상대위험도 구하기

In [ ]:
mp = smf.glm(f_adj, df, family=fam.Poisson()).fit(cov_type="HC0")
ml = smf.glm(f_adj, df, family=fam.Binomial()).fit()
for name, r in [("aRR, modified Poisson", mp), ("aOR, logistic", ml)]:
    lo, hi = np.exp(r.conf_int().loc["smoke"])
    est = np.exp(r.params["smoke"])
    print(f"{name:22s}: {est:.2f} (95% CI {lo:.2f} to {hi:.2f})")
print("max fitted p (Poisson):", round(mp.fittedvalues.max(), 3))

## c. 교란과 층화 분석

### 셀 17. 인종은 교란변수의 조건을 갖추었나

In [ ]:
g = df.groupby("race_f", observed=True)
g_non = df[df["smoke"] == 0].groupby("race_f", observed=True)
pd.DataFrame({
    "n": g.size(),
    "smoker %": g["smoke"].mean() * 100,
    "LBW % (all)": g["low"].mean() * 100,
    "LBW % (nonsmokers)": g_non["low"].mean() * 100,
}).round(1)

### 셀 18. 인종별 2×2 표와 층별 오즈비

In [ ]:
tables = []
for race, sub in df.groupby("race_f", observed=True):
    t = pd.crosstab(sub["smoke"], sub["low"]).loc[[1, 0], [1, 0]]
    tables.append(t.to_numpy())
    t2 = Table2x2(t.to_numpy())
    lo, hi = t2.oddsratio_confint()
    print(f"{race:5} {t.to_numpy().tolist()}  "
          f"OR = {t2.oddsratio:.2f} ({lo:.2f} to {hi:.2f})")

### 셀 19. Mantel–Haenszel 통합 오즈비와 두 가지 검정

In [ ]:
from statsmodels.stats.contingency_tables import StratifiedTable

st = StratifiedTable(tables)   # 2×2 표 3개의 목록
print(st.summary())

### 셀 20. 보정 전과 보정 후 비교

In [ ]:
m_race = smf.logit("low ~ smoke + C(race)", df).fit(disp=0)
b = m_race.params["smoke"]
b_lo, b_hi = m_race.conf_int().loc["smoke"]
rows = [["crude", t22.oddsratio, *t22.oddsratio_confint()],
        ["Mantel-Haenszel", st.oddsratio_pooled,
         *st.oddsratio_pooled_confint()],
        ["logistic + C(race)", np.exp(b), np.exp(b_lo), np.exp(b_hi)]]
pd.DataFrame(rows, columns=["method", "OR", "2.5%", "97.5%"]).round(2)

### 셀 21. 층별 오즈비와 통합 오즈비 그래프

In [ ]:
labels = ["White", "Black", "Other", "Crude", "Mantel-Haenszel"]
t_all = [Table2x2(t) for t in tables] + [t22]
est = [t.oddsratio for t in t_all] + [st.oddsratio_pooled]
ci = [t.oddsratio_confint() for t in t_all]
ci += [st.oddsratio_pooled_confint()]
err = np.array([[e - l, h - e] for e, (l, h) in zip(est, ci)]).T
ypos = np.arange(len(labels))[::-1]
plt.figure(figsize=(6, 2.8))
plt.errorbar(est, ypos, xerr=err, fmt="s", color="k", capsize=3)
plt.axvline(1, ls="--", color="gray")
plt.xscale("log")
plt.xticks([0.5, 1, 2, 5, 10, 20], [0.5, 1, 2, 5, 10, 20])
plt.minorticks_off()
plt.yticks(ypos, labels)
plt.xlabel("Odds ratio for smoking (log scale)")
plt.show()

## d. 가변수와 다변수 분석

### 셀 22. 인종을 숫자 그대로 넣으면

In [ ]:
m_num = smf.logit("low ~ smoke + race", df).fit(disp=0)
m_cat = smf.logit("low ~ smoke + C(race, Treatment(reference=1))",
                  df).fit(disp=0)
print(m_num.params.round(3), "\n")
print(m_cat.params.round(3), "\n")
print(f"llf: numeric race {m_num.llf:.3f}, dummies {m_cat.llf:.3f}")

### 셀 23. 기준범주 바꾸기

In [ ]:
m_ref3 = smf.logit("low ~ smoke + C(race, Treatment(reference=3))",
                   df).fit(disp=0)
print(np.exp(m_ref3.params).round(3), "\n")
print(f"llf = {m_ref3.llf:.3f}")

### 셀 24. 인종 전체의 P: 결합 검정

In [ ]:
print(m_cat.wald_test_terms())
m_smoke = smf.logit("low ~ smoke", df).fit(disp=0)
lr = 2 * (m_cat.llf - m_smoke.llf)
print(f"\nLR test for race: chi2 = {lr:.3f}, df = 2, "
      f"P = {stats.chi2.sf(lr, 2):.4f}")

### 셀 25. 단변수 오즈비: 변수마다 따로 적합

In [ ]:
def or_ci(model):
    t = np.exp(model.conf_int())
    t.columns = ["lo", "hi"]
    t.insert(0, "OR", np.exp(model.params))
    return t.drop(index="Intercept")

terms = ["smoke", "C(race)", "age", "lwt_kg", "ptl_any", "ht"]
uni = pd.concat([or_ci(smf.logit(f"low ~ {t}", df).fit(disp=0))
                 for t in terms])
uni.round(2)

### 셀 26. Table 2 만들기

In [ ]:
full = smf.logit("low ~ " + " + ".join(terms), df).fit(disp=0)
multi = or_ci(full).loc[uni.index]      # 단변수 표와 같은 행 순서
fmt = lambda t: [f"{r.OR:.2f} ({r.lo:.2f}-{r.hi:.2f})"
                 for r in t.itertuples()]
table2 = pd.DataFrame({"Univariable OR (95% CI)": fmt(uni),
                       "Multivariable OR (95% CI)": fmt(multi)},
                      index=uni.index)
print("n =", int(full.nobs), " events =", int(df["low"].sum()),
      " coefficients =", len(full.params) - 1)
table2

### 셀 27. 다변수 모형의 변수별 결합 검정

In [ ]:
print(full.wald_test_terms())

### 셀 28. 행 이름을 논문용으로 바꾸기

In [ ]:
table2.index = ["Smoking during pregnancy",
                "Race: Black (vs White)", "Race: Other (vs White)",
                "Age, per year", "Weight before pregnancy, per kg",
                "Previous preterm labor", "History of hypertension"]
table2

## e. 과제 정답

**과제 1.** 셀 6의 NNH 행(6.526, 95% CI 3.444 to 62.221)을 한 문장으로 옮기고, 이 숫자가 말해 주지 않는 것을 두 가지 이상 쓰세요. 셀 20의 인종 보정 모형과 셀 26의 다변수 모형으로 표준화한 위험차와 NNH를 구해 비교하세요.

### 셀 29. 과제 1 정답. NNH를 보정 모형으로 다시 구하기

In [ ]:
print(f"crude          : risk {p1:.3f} vs {p0:.3f}, "
      f"RD = {p1 - p0:.3f}, NNH = {1 / (p1 - p0):.1f}")
for name, model in [("race only", m_race), ("all covariates", full)]:
    r1 = model.predict(df.assign(smoke=1)).mean()   # 모두 흡연했다면
    r0 = model.predict(df.assign(smoke=0)).mean()   # 모두 비흡연이라면
    print(f"{name:15s}: risk {r1:.3f} vs {r0:.3f}, "
          f"RD = {r1 - r0:.3f}, NNH = {1 / (r1 - r0):.1f}")

**과제 2.** 저체중아의 기준을 2,500 g에서 2,000 g과 3,000 g으로 바꿔 셀 6처럼 흡연의 상대위험도와 오즈비를 다시 구하세요. 기준에 따라 두 지표의 차이가 왜 달라지는지 설명하세요.

### 셀 30. 과제 2 정답. 저체중아의 기준을 바꾸면

In [ ]:
rows = []
for cut in [2000, 2500, 3000]:
    y_cut = (df["bwt"] < cut).astype(int)            # 결과를 다시 정의
    t = pd.crosstab(df["smoke"], y_cut).loc[[1, 0], [1, 0]].to_numpy()
    t2 = Table2x2(t)
    r_s, r_n = t[0, 0] / t[0].sum(), t[1, 0] / t[1].sum()  # 흡연, 비흡연 위험
    rows.append([cut, y_cut.sum(), y_cut.mean(), r_s, r_n,
                 t2.riskratio, t2.oddsratio, t2.oddsratio / t2.riskratio])
cols = ["cut", "events", "prop", "risk_s", "risk_n", "RR", "OR", "OR/RR"]
pd.DataFrame(rows, columns=cols).round(3)

**과제 3.** 동료가 인종별 오즈비(셀 18)를 보고 "흡연은 백인 산모에서만 저체중아와 유의하게 관련되었고(OR 5.76, 95% CI 1.78–18.60), 흑인과 기타 산모에서는 관련이 없었다. 흡연의 영향은 인종에 따라 다르다"라고 썼습니다. 이 결론에서 틀린 곳 두 가지를 찾고, 인종에 따라 효과가 다른지 직접 검정하세요.

### 셀 31. 과제 3 정답. 인종에 따라 흡연의 효과가 다른가

In [ ]:
m_int = smf.logit("low ~ smoke * C(race)", df).fit(disp=0)  # 교호작용 항 포함
lr_int = 2 * (m_int.llf - m_race.llf)               # 교호작용 항이 없는 모형과 비교
print(f"P for interaction (LR, df = 2): chi2 = {lr_int:.2f}, "
      f"P = {stats.chi2.sf(lr_int, 2):.3f}")
print(np.exp(m_int.params).round(2), "\n")
risk = df.pivot_table(index="race_f", columns="smoke", values="low",
                      aggfunc="mean", observed=True)
print((risk[1] - risk[0]).round(3))                  # 인종별 위험차